Celda 1: Importación de librerías y configuración de entorno

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path

# Configuración de dispositivo (GPU si está disponible, sino CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

# Rutas del proyecto
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "dataset_burbujas"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparámetros
BATCH_SIZE = 32
EPOCHS = 10 # Ideal para 260 imágenes; puedes subirlo cuando tengas las 1300
LEARNING_RATE = 0.001

Celda 2: Carga de Datos y Transformaciones

In [ ]:
# Transformaciones: Escala de grises, redimensionamiento a 64x64 y conversión a Tensor
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]) # Normalización básica [-1, 1]
])

# Cargar dataset completo
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# Guardar el mapeo de clases (ImageFolder ordena alfabéticamente: 0: EMPTY, 1: GHOST, 2: MARKED)
class_names = full_dataset.classes
class_to_idx = full_dataset.class_to_idx
print(f"Clases detectadas: {class_to_idx}")

# Guardar labels.json para la fase de producción
labels_path = MODELS_DIR / "labels.json"
with open(labels_path, "w", encoding="utf-8") as f:
    # Invertimos el diccionario para guardar { "0": "EMPTY", ... }
    json.dump({str(v): k for k, v in class_to_idx.items()}, f, indent=4)
print(f"Archivo de etiquetas guardado en: {labels_path}")

# Dividir el dataset (80% entrenamiento, 20% validación)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Imágenes de entrenamiento: {len(train_dataset)}")
print(f"Imágenes de validación: {len(val_dataset)}")

Celda 3: Definición de la Arquitectura CNN

In [ ]:
class BubbleClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super(BubbleClassifier, self).__init__()
        
        # Bloque 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 16 x 32 x 32
        
        # Bloque 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 32 x 16 x 16
        
        # Bloque 3
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # Salida: 64 x 8 x 8
        
        # Clasificador Densa
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features=64 * 8 * 8, out_features=128)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5) # Previene sobreajuste
        self.fc2 = nn.Linear(in_features=128, out_features=num_classes)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        
        x = self.flatten(x)
        x = self.relu4(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Instanciar el modelo
model = BubbleClassifier(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)

Celda 4: Bucle de Entrenamiento y Validación

In [ ]:
best_val_acc = 0.0
model_save_path = MODELS_DIR / "bubble_classifier_v1.pt"

print("Iniciando entrenamiento...")

for epoch in range(EPOCHS):
    # --- Fase de Entrenamiento ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() # Reiniciar gradientes
        
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels)
        
        loss.backward() # Backward pass
        optimizer.step() # Optimización
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    
    # --- Fase de Validación ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # No necesitamos calcular gradientes aquí
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_loss = val_loss / total_val
    val_acc = correct_val / total_val
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Guardar el mejor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), model_save_path)
        print(f"  -> Nuevo mejor modelo guardado (Val Acc: {val_acc:.4f})")

print(f"\nEntrenamiento completado. Mejor modelo exportado a: {model_save_path}")

Celda 5: Test de Inferencia (Simulación de Fase de Producción)

In [ ]:
import torch.nn.functional as F

# Cargar el modelo guardado
model.load_state_dict(torch.load(model_save_path))
model.eval()

# Tomamos un lote de validación para probar
dataiter = iter(val_loader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    # Aplicar Softmax para obtener probabilidades [0, 1]
    probabilities = F.softmax(outputs, dim=1)
    
    # Obtener la clase con mayor probabilidad y su valor (confidence)
    confidence, predicted = torch.max(probabilities, 1)

print("\n--- Resultados de Inferencia de Prueba ---")
for i in range(5): # Mostrar los primeros 5 resultados
    real_class = class_names[labels[i].item()]
    pred_class = class_names[predicted[i].item()]
    conf_val = confidence[i].item()
    
    print(f"Muestra {i+1}: Real: {real_class:<8} | Predicción: {pred_class:<8} | Confidence: {conf_val:.4f}")

Celda 6: Métricas de Evaluación Formal y Matriz de Confusión 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print("Generando métricas formales sobre el set de validación...")

# Asegurarnos de cargar el mejor modelo
model.load_state_dict(torch.load(model_save_path))
model.eval()

all_labels = []
all_preds = []

# Recorrer TODO el set de validación
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        # Guardar resultados en listas de CPU para sklearn
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

# 1. Reporte de Clasificación (Precision, Recall, F1-Score)
print("\n--- Reporte de Clasificación ---")
report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
print(report)

# 2. Matriz de Confusión
cm = confusion_matrix(all_labels, all_preds)

# Configurar el gráfico de la matriz
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusión - Bubble Classifier')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.tight_layout()

# Guardar la imagen para tu documento de P5
metrics_dir = PROJECT_ROOT / "docs" / "evidencia_modelo"
metrics_dir.mkdir(parents=True, exist_ok=True)
cm_path = metrics_dir / "matriz_confusion.png"
plt.savefig(cm_path, dpi=300)
print(f"Matriz de confusión exportada exitosamente a: {cm_path}")

plt.show()